# Question 28: Build a Complete LLM Inference Engine

### Difficulty: Expert

### Problem Statement

In this capstone exercise, you will build a **minimal but complete** LLM inference engine that integrates several components you have learned:

1. **A simple transformer model** (2-layer, with self-attention and feed-forward layers)
2. **KV cache** for efficient autoregressive generation
3. **Top-p (nucleus) sampling** for controllable text generation
4. **Basic batching** to handle multiple concurrent requests

The engine accepts text prompts (using a simple character-level tokenizer), generates completions, and supports multiple concurrent requests.

This mirrors the core of what production inference engines (vLLM, TGI, TensorRT-LLM) do, stripped down to its essentials.

---

### Requirements

1. **`SimpleTokenizer`**: Character-level tokenizer.
   - `encode(text) -> List[int]`: Convert text to token IDs.
   - `decode(ids) -> str`: Convert token IDs back to text.
   - Reserve ID 0 for `<pad>` and ID 1 for `<eos>`.

2. **`MiniTransformer(nn.Module)`**: A 2-layer transformer.
   - Token embedding + positional embedding.
   - Each layer: multi-head self-attention (with KV cache support) + feed-forward network.
   - Layer normalization (pre-norm style).
   - Output projection to vocabulary logits.

3. **`InferenceEngine`**: Ties everything together.
   - `generate(prompts, max_new_tokens, temperature, top_p)`: Accept a list of text prompts, tokenize, generate completions, decode back to text.
   - Uses KV cache internally for efficient generation.
   - Implements top-p sampling.

4. **Validation**: Generate text from multiple prompts simultaneously, showing the engine handles them correctly.

---

### Constraints

- Use only PyTorch operations.
- The transformer must use pre-norm (LayerNorm before attention/FFN, not after).
- Top-p sampling: sort probabilities descending, compute cumulative sum, zero out tokens beyond the cumulative threshold `p`, renormalize, then sample.
- The model does NOT need to produce coherent text (it is untrained). The goal is correct architecture and inference logic.

---

### Company Tags

Perplexity, Together AI, Anyscale, Fireworks AI

---

<details>
  <summary>Hint</summary>

  - For the tokenizer, use `ord(c) + 2` as the token ID for character `c` (offset by 2 to reserve 0 and 1 for special tokens). For decoding, `chr(id - 2)`.
  - For KV cache in the transformer: each layer gets its own KVCache. Pass a list of caches (one per layer) through the forward pass.
  - For top-p sampling: `sorted_probs, sorted_indices = torch.sort(probs, descending=True)`. Then `cumsum = torch.cumsum(sorted_probs, dim=-1)`. Mask out where `cumsum - sorted_probs >= top_p`.
  - For batching: pad all prompts to the same length before passing to the model. Use attention masking to ignore padding.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import List, Optional, Tuple

In [ ]:
# Configuration
torch.manual_seed(42)

d_model = 64
num_heads = 4
d_ff = 128              # Feed-forward hidden dimension
num_layers = 2
max_seq_len = 256
temperature = 0.8
top_p = 0.9

device = "cpu"
print(f"Config: d_model={d_model}, heads={num_heads}, layers={num_layers}, d_ff={d_ff}")

In [ ]:
class SimpleTokenizer:
    """
    Character-level tokenizer.
    
    Special tokens:
        0 = <pad>
        1 = <eos>
    Regular characters: ord(c) + 2
    """
    
    def __init__(self):
        # TODO: Define pad_id, eos_id, and compute vocab_size
        # vocab_size should cover all printable ASCII + 2 special tokens
        ...
    
    def encode(self, text):
        """Convert text to list of token IDs."""
        # TODO: Map each character to ord(c) + 2
        ...
    
    def decode(self, ids):
        """Convert list of token IDs back to text. Skip pad/eos."""
        # TODO: Map each ID back to chr(id - 2), skip special tokens
        ...


print("SimpleTokenizer defined. Fill in the TODO sections above.")

In [ ]:
class KVCache:
    """KV Cache for a single attention layer."""
    
    def __init__(self):
        # TODO: Initialize empty cache
        ...
    
    def update(self, new_k, new_v):
        """Append new K/V and return full cached K/V."""
        # TODO
        ...
    
    def reset(self):
        # TODO
        ...


class MultiHeadAttention(nn.Module):
    """Multi-head attention with KV cache support."""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        # TODO: Initialize Q, K, V, output projections
        # Store d_model, num_heads, d_head
        ...
    
    def forward(self, x, kv_cache=None):
        """
        Args:
            x: (B, S, d_model)
            kv_cache: Optional KVCache
        Returns:
            output: (B, S, d_model)
        """
        # TODO: Implement attention with optional KV cache
        # Same logic as Question 12
        ...


class FeedForward(nn.Module):
    """Simple feed-forward network: Linear -> GELU -> Linear."""
    
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO: Two linear layers with GELU activation in between
        ...
    
    def forward(self, x):
        # TODO
        ...


class TransformerBlock(nn.Module):
    """Single transformer block: pre-norm attention + pre-norm FFN."""
    
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        # TODO: Initialize LayerNorms, MultiHeadAttention, FeedForward
        ...
    
    def forward(self, x, kv_cache=None):
        """
        Pre-norm transformer block:
            x = x + Attention(LayerNorm(x))
            x = x + FFN(LayerNorm(x))
        """
        # TODO
        ...


class MiniTransformer(nn.Module):
    """
    Minimal 2-layer transformer for text generation.
    
    Components:
    - Token embedding + positional embedding
    - N transformer blocks (each with attention + FFN)
    - Final layer norm
    - Output projection to vocab logits
    """
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len):
        super().__init__()
        # TODO: Initialize embeddings, transformer blocks, final norm, output projection
        ...
    
    def forward(self, input_ids, kv_caches=None):
        """
        Args:
            input_ids: (B, S) token indices
            kv_caches: Optional list of KVCache (one per layer)
        Returns:
            logits: (B, S, vocab_size)
        """
        # TODO:
        # 1. Token embedding + positional embedding
        #    - For cached generation, position offset = cache length
        # 2. Pass through each transformer block (with its KV cache)
        # 3. Final layer norm
        # 4. Output projection
        ...


print("Model components defined. Fill in the TODO sections above.")

In [ ]:
def top_p_sample(logits, temperature=1.0, top_p=0.9):
    """
    Top-p (nucleus) sampling.
    
    Args:
        logits: (B, vocab_size) raw logits for next token
        temperature: Sampling temperature
        top_p: Cumulative probability threshold
    
    Returns:
        sampled_tokens: (B, 1) sampled token IDs
    """
    # TODO:
    # 1. Apply temperature scaling
    # 2. Compute softmax probabilities
    # 3. Sort probabilities in descending order
    # 4. Compute cumulative sum
    # 5. Create mask: keep tokens where (cumsum - current_prob) < top_p
    # 6. Zero out masked probabilities and renormalize
    # 7. Sample from the filtered distribution
    ...


class InferenceEngine:
    """
    Complete inference engine tying together:
    - Tokenizer
    - Transformer model with KV cache
    - Top-p sampling
    - Basic batching
    """
    
    def __init__(self, model, tokenizer, device="cpu"):
        # TODO: Store model, tokenizer, device
        ...
    
    def generate(self, prompts, max_new_tokens=50, temperature=0.8, top_p=0.9):
        """
        Generate completions for a list of text prompts.
        
        Args:
            prompts: List of text strings
            max_new_tokens: Maximum tokens to generate
            temperature: Sampling temperature
            top_p: Nucleus sampling threshold
        
        Returns:
            List of generated text strings (prompt + completion)
        """
        # TODO:
        # 1. Tokenize all prompts
        # 2. Pad to same length (left-pad or right-pad with pad_id)
        # 3. Create KV caches (one per layer)
        # 4. Prefill: run model on full prompt to populate KV cache
        # 5. Decode loop:
        #    a. Get logits for the last token position
        #    b. Apply top-p sampling
        #    c. Append new tokens
        #    d. Feed only the new token through the model (using KV cache)
        #    e. Stop if all sequences have generated EOS or hit max length
        # 6. Decode token IDs back to text
        ...


print("InferenceEngine defined. Fill in the TODO sections above.")

In [ ]:
# ============================================================
# Validation: Generate from multiple prompts
# ============================================================

torch.manual_seed(42)

# Initialize tokenizer
tokenizer = SimpleTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")

# Test tokenizer roundtrip
test_text = "Hello, World!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"Tokenizer roundtrip: '{test_text}' -> {encoded} -> '{decoded}'")
assert decoded == test_text, "Tokenizer roundtrip failed!"
print()

# Initialize model and engine
model = MiniTransformer(
    vocab_size=tokenizer.vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=num_layers,
    max_seq_len=max_seq_len
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print()

engine = InferenceEngine(model, tokenizer, device=device)

# Generate from multiple prompts
prompts = [
    "The quick brown ",
    "In a galaxy ",
    "def fibonacci(",
    "Once upon a ",
]

print("Generating completions...")
print("=" * 60)

with torch.no_grad():
    outputs = engine.generate(
        prompts,
        max_new_tokens=40,
        temperature=temperature,
        top_p=top_p
    )

for i, (prompt, output) in enumerate(zip(prompts, outputs)):
    generated_part = output[len(prompt):]
    print(f"\nPrompt {i}: '{prompt}'")
    print(f"Generated: '{generated_part}'")
    print(f"Full: '{output}'")

print("\n" + "=" * 60)
print("Inference engine validation complete.")
print("(Note: output is random since the model is untrained.)")